In [1]:
import sys
import os
import pickle
import signal
signal.signal(signal.SIGINT, signal.SIG_DFL)

from pyscf.pbc.df.df import CDERIArray
from pyscf import lib
from pyscf.pbc import gto, scf, df
import numpy as np

import pickle
import h5py

# Obtain GDF data

In [ ]:
nk = 2
chk = f"data_GDF/GDF_diamond_{nk}x{nk}x{nk}_gth-dzvp_ke40.0.chk"

def cderi_block_to_3D(cderi, ki, kj):
    nao = cderi.nao
    blk = cderi[ki, kj]
    
    if blk.shape[-1] == nao**2:
        return blk.reshape(-1, nao, nao)
    elif blk.shape[-1] == nao * (nao + 1)//2:
        return lib.unpack_tril(blk).reshape(-1, nao, nao)
    else:
        raise ValueError(f"Unexpected AO-pair dimension {blk.shape[-1]} for nao={nao}")

with h5py.File(chk, "r") as f:
    cderi = CDERIArray(f)
    
    nkpts = cderi.nkpts
    nao = cderi.nao
    naux = cderi.naux
    
    L = np.empty((nkpts, nkpts, naux, nao, nao), dtype=complex)
    
    for ki in range(nkpts):
        for kj in range(nkpts):
            L[ki, kj] = cderi_block_to_3D(cderi, ki, kj)

# Obtain vertical norm

In [47]:
alpha = 0
for Q in range(nkpts):
    pairs = []
    for kp in range(nkpts):
        for kq in range(nkpts):
            Q_cur = (kq - kp) % nkpts
            if Q_cur == Q:
                pairs.append((kp, kq)) 
    npairs = len(pairs)
    for i in range(naux):
        M = np.zeros((nkpts * nao, nkpts * nao), dtype=complex)
        for (kp, kq) in pairs:
            r = slice(kp*nao, (kp+1)*nao)
            c = slice(kq*nao, (kq+1)*nao)
            M[r, c] = L[kp, kq, i, :, :]
        norm = np.linalg.norm(M, ord=2)
        alpha += norm ** 2
        print(f"({Q}, {i}) finished.")

(0, 0) finished.
(0, 1) finished.
(0, 2) finished.
(0, 3) finished.
(0, 4) finished.
(0, 5) finished.
(0, 6) finished.
(0, 7) finished.
(0, 8) finished.
(0, 9) finished.
(0, 10) finished.
(0, 11) finished.
(0, 12) finished.
(0, 13) finished.
(0, 14) finished.
(0, 15) finished.
(0, 16) finished.
(0, 17) finished.
(0, 18) finished.
(0, 19) finished.
(0, 20) finished.
(0, 21) finished.
(0, 22) finished.
(0, 23) finished.
(0, 24) finished.
(0, 25) finished.
(0, 26) finished.
(0, 27) finished.
(0, 28) finished.
(0, 29) finished.
(0, 30) finished.
(0, 31) finished.
(0, 32) finished.
(0, 33) finished.
(0, 34) finished.
(0, 35) finished.
(0, 36) finished.
(0, 37) finished.
(0, 38) finished.
(0, 39) finished.
(0, 40) finished.
(0, 41) finished.
(0, 42) finished.
(0, 43) finished.
(0, 44) finished.
(0, 45) finished.
(0, 46) finished.
(0, 47) finished.
(0, 48) finished.
(0, 49) finished.
(0, 50) finished.
(0, 51) finished.
(0, 52) finished.
(0, 53) finished.
(0, 54) finished.
(0, 55) finished.
(0

In [48]:
alpha

np.float64(498.3477614755232)

# Obtain horizontal norm

In [55]:
M = L.transpose(0, 1, 3, 4, 2).reshape(nkpts**2 * nao**2, naux)
np.linalg.norm(M, ord=2)**2

np.float64(862.5358227195056)

# Obtain HF data

In [63]:
nk = 2
mf_file = f"data_GDF/SCF_diamond_{nk}x{nk}x{nk}_gth-dzvp_ke40.0.pkl"

with open(mf_file, "rb") as f:
    mf = pickle.load(f)

print(type(mf))
print(mf)

<class 'pyscf.pbc.scf.khf.KRHF'>


In [94]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
occ_tol = 1E-8

nkpts = len(mo_coeff)

Cocc = []
Cvir = []
Sovlp = []

for k in range(nkpts):
    occ_mask = mo_occ[k] > occ_tol
    vir_mask = mo_occ[k] <= occ_tol

    Cocc.append(mo_coeff[k][:, occ_mask])
    Cvir.append(mo_coeff[k][:, vir_mask])
    Sovlp.append(mf.get_ovlp()[k])

nv = Cvir[0].shape[1]
no = Cocc[0].shape[1]

In [95]:
alpha = 0
for Q in range(nkpts):
    pairs = []
    for kp in range(nkpts):
        for kq in range(nkpts):
            Q_cur = (kq - kp) % nkpts
            if Q_cur == Q:
                pairs.append((kp, kq)) 
    npairs = len(pairs)
    for i in range(naux):
        M_oo = np.zeros((nkpts * no, nkpts * no), dtype=complex)
        M_vv = np.zeros((nkpts * nv, nkpts * nv), dtype=complex)
        for (kp, kq) in pairs:
            r = slice(kp*no, (kp+1)*no)
            c = slice(kq*no, (kq+1)*no)
            M_oo[r, c] = Cocc[kp].conj().T @ L[kp, kq, i, :, :] @ Cocc[kq]
            
            r = slice(kp*nv, (kp+1)*nv)
            c = slice(kq*nv, (kq+1)*nv)
            M_vv[r, c] = Cvir[kp].conj().T @ L[kp, kq, i, :, :] @ Cvir[kq]
            
        norm_oo = np.linalg.norm(M_oo, ord=2)
        norm_vv = np.linalg.norm(M_vv, ord=2)
        alpha += norm_oo * norm_vv
        print(f"({Q}, {i}) finished.")

(0, 0) finished.
(0, 1) finished.
(0, 2) finished.
(0, 3) finished.
(0, 4) finished.
(0, 5) finished.
(0, 6) finished.
(0, 7) finished.
(0, 8) finished.
(0, 9) finished.
(0, 10) finished.
(0, 11) finished.
(0, 12) finished.
(0, 13) finished.
(0, 14) finished.
(0, 15) finished.
(0, 16) finished.
(0, 17) finished.
(0, 18) finished.
(0, 19) finished.
(0, 20) finished.
(0, 21) finished.
(0, 22) finished.
(0, 23) finished.
(0, 24) finished.
(0, 25) finished.
(0, 26) finished.
(0, 27) finished.
(0, 28) finished.
(0, 29) finished.
(0, 30) finished.
(0, 31) finished.
(0, 32) finished.
(0, 33) finished.
(0, 34) finished.
(0, 35) finished.
(0, 36) finished.
(0, 37) finished.
(0, 38) finished.
(0, 39) finished.
(0, 40) finished.
(0, 41) finished.
(0, 42) finished.
(0, 43) finished.
(0, 44) finished.
(0, 45) finished.
(0, 46) finished.
(0, 47) finished.
(0, 48) finished.
(0, 49) finished.
(0, 50) finished.
(0, 51) finished.
(0, 52) finished.
(0, 53) finished.
(0, 54) finished.
(0, 55) finished.
(0

In [96]:
alpha

np.float64(18.049261843616065)